# Metadata Filtering

In [3]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

In [1]:
import pandas as pd

document_df = pd.read_csv("data/documents_meta.csv")
document_df

,doc_id,title,content,author,category
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...",박민준,여행;제주;관광
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기...",이서연,음식;비빔밥;역사
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet...",최유나,연예;음악;대중문화
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...,정하늘,역사;문화;문자
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...,정하늘,역사;군사;전쟁
5,D6,2024년 기후 변화 종합 보고서,"2024년 전 지구 평균 기온은 산업화 이전 대비 약 1.2℃ 상승했으며, 해수면 ...",한지민,환경;기후;정책
6,D7,AI 기술 동향 및 윤리,"최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발...",김도윤,AI;기술;윤리
7,D8,서울 지하철 이용 가이드,"서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로...",박민준,교통;서울;생활
8,D9,판소리 “춘향가” 서사 구조,"판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가...",오세린,문화;음악;전통예술
9,D10,한국 축구 대표팀 주요 기록,"한국 축구 대표팀은 2002 한일 월드컵 4강 진출, 2012 런던 올림픽 동메달 ...",강민재,스포츠;축구;기록


In [4]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()
print(pc.list_indexes().names())

PINECONE_META_INDEX_NAME = 'adv-meta-rag'

if PINECONE_META_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_META_INDEX_NAME,
        dimension=PINECONE_INDEX_DEMENSION,
        metric=PINECONE_INDEX_METRIc,
        spec=ServerlessSpec(
            region=PINECONE_INDEX_REGION,
            cloud=PINECONE_INDEX_CLOUD
        )
    )
    print(f'{PINECONE_META_INDEX_NAME} index 생성 완료')
else:
    print(f'{PINECONE_META_INDEX_NAME} index가 이미 존재합니다.')

['adv-comp-rag', 'adv-rag', 'winemeg-review-data']
adv-meta-rag index 생성 완료


In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터 스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_META_INDEX_NAME,
    embedding=embeddings
)

In [6]:
from langchain_core.documents import Document

docs = []
ids = []

for idx,row in document_df.iterrows():
    doc_id = row['doc_id']
    content = row['content']
    
    doc = Document(
        page_content=content,
        metadata = {
            "doc_id":doc_id,
            "author":row['author'],
            "category":row['category'].split(";")
        }
    )

    docs.append(doc)
    ids.append(doc_id)

vector_store.add_documents(
    documents=docs,
    ids=ids
)

print("Pinecone 문서 저장 완료")
print(vector_store._index.describe_index_stats())

Pinecone 문서 저장 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 30}},
 'total_vector_count': 30,
 'vector_type': 'dense'}


## 메타데이터 필터링을 이용한 벡터 검색

LangChain의 `vector_store.similarity_search`에서 `filter`를 사용하면 문서의 메타데이터(metadata)를 조건으로 검색 결과를 제한할 수 있다.

벡터 검색은 기본적으로 사용자의 질문과 문서 내용의 의미적 유사도를 기준으로 문서를 찾는다. 하지만 실제 서비스에서는 단순히 내용이 비슷한 문서를 찾는 것만으로는 부족한 경우가 많다. 예를 들어 특정 작성자의 문서만 검색하거나, 특정 카테고리에 속한 문서만 검색하거나, 특정 조건을 만족하는 문서만 검색해야 할 수 있다.

이때 메타데이터 필터링을 사용한다.

메타데이터 필터링은 검색 결과의 점수를 직접 높이는 기능이 아니라, 검색 대상이 되는 문서의 범위를 제한하는 기능이다. 즉, 먼저 metadata filter 조건을 만족하는 문서만 후보로 남고, 그 후보 문서 안에서 query와 벡터 유사도가 높은 문서가 상위 결과로 반환된다.

```python
results = vector_store.similarity_search(
    query="검색할 내용",
    k=5,
    filter={
        "category": "역사"
    }
)

### Pinecone의 metadata filter

Pinecone은 metadata filter에서 MongoDB 스타일과 유사한 연산자를 지원한다. 따라서 특정 값과 정확히 일치하는 문서뿐만 아니라, 여러 값 중 하나에 포함되는 문서, 특정 값보다 크거나 작은 문서, 여러 조건을 동시에 만족하는 문서 등을 검색할 수 있다.

Pinecone에서 지원하는 주요 연산자는 다음과 같다.

| 연산자       | 의미                   | 예시                                                  |
| --------- | -------------------- | --------------------------------------------------- |
| `$eq`     | 같음                   | `{"author": {"$eq": "김철수"}}`                        |
| `$ne`     | 같지 않음                | `{"author": {"$ne": "김철수"}}`                        |
| `$gt`     | 큼                    | `{"year": {"$gt": 2020}}`                           |
| `$lt`     | 작음                   | `{"year": {"$lt": 2020}}`                           |
| `$gte`    | 크거나 같음               | `{"year": {"$gte": 2020}}`                          |
| `$lte`    | 작거나 같음               | `{"year": {"$lte": 2020}}`                          |
| `$in`     | 지정한 값 중 하나에 포함       | `{"category": {"$in": ["역사", "기술"]}}`               |
| `$nin`    | 지정한 값 중 어디에도 포함되지 않음 | `{"category": {"$nin": ["연예", "영화"]}}`              |
| `$exists` | 특정 metadata 필드 존재 여부 | `{"author": {"$exists": true}}`                     |
| `$and`    | 여러 조건을 모두 만족         | `{"$and": [{"author": "김철수"}, {"category": "검색"}]}` |
| `$or`     | 여러 조건 중 하나 이상 만족     | `{"$or": [{"author": "김철수"}, {"category": "AI"}]}`  |

In [7]:
# 예시 1. 특정 카테고리에 해당하는 문서만 검색
query = '훈민정음'

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        'category':'역사'
    }
)

for doc in results:
    print(f"{doc.metadata['doc_id']}:{doc.metadata['category']}")
    print(doc.page_content)
    print()

D4:['역사', '문화', '문자']
세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입니다. 그가 훈민정음을 만든 배경에는 백성들의 문맹 문제 해결과 국가 통치 효율화가 있었습니다. 세종대왕의 업적은 한국 문화와 문자 체계에 지대한 영향을 미쳤으며, 훈민정음 해례본은 유네스코 세계기록유산으로 등재되었습니다.

D5:['역사', '군사', '전쟁']
이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133척의 왜선을 격파하면서 크게 승리했습니다. 전술적인 배 배치(학익진)와 기상·해류를 활용한 전략은 전투 역사에 길이 남을 전술입니다. 이순신의 업적은 한국 해군 전통과 군사 전략 연구에서 핵심 사례로 다뤄집니다.

D2:['음식', '비빔밥', '역사']
비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기·계란 등)을 올리고 고추장이나 간장을 섞어 먹습니다. 전주 비빔밥은 고명 종류가 다양하고 전주식 고추장을 쓰며, 잔치용으로도 유명합니다. 진주 비빔밥은 고기·회·나물 등을 섞어 더욱 풍부한 식감을 제공합니다. 두 지역 모두 역사적 배경과 재료 구성이 달라 맛과 풍미가 다릅니다.



In [ ]:
# 예시 2. 여러 카테고리 중 하나에 해당하는 문서만 검색
query = '훈민정음'

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        'category':{'$in' : ['역사','기술']}
    }
)

for doc in results:
    print(f"{doc.metadata['doc_id']}:{doc.metadata['category']}")
    print(doc.page_content)
    print()

D4:['역사', '문화', '문자']
세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입니다. 그가 훈민정음을 만든 배경에는 백성들의 문맹 문제 해결과 국가 통치 효율화가 있었습니다. 세종대왕의 업적은 한국 문화와 문자 체계에 지대한 영향을 미쳤으며, 훈민정음 해례본은 유네스코 세계기록유산으로 등재되었습니다.

D5:['역사', '군사', '전쟁']
이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133척의 왜선을 격파하면서 크게 승리했습니다. 전술적인 배 배치(학익진)와 기상·해류를 활용한 전략은 전투 역사에 길이 남을 전술입니다. 이순신의 업적은 한국 해군 전통과 군사 전략 연구에서 핵심 사례로 다뤄집니다.

D7:['AI', '기술', '윤리']
최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발이 활발합니다. GPT 계열(예: GPT-4, GPT-4o-mini)은 자연어 생성·이해 능력이 뛰어나며, DALL·E, Stable Diffusion은 이미지 생성, CLIP 등은 이미지·텍스트 융합 모델로 주목받고 있습니다. 또한 AI 윤리 이슈로는 데이터 편향, 프라이버시 침해, 자율성 문제 등이 논의되고 있습니다.

D27:['검색', '벡터DB', '기술']
ChromaDB와 Qdrant는 벡터 검색 라이브러리로, ChromaDB는 오픈소스 벡터 DB로 간단한 파이썬 인터페이스를 제공하며, Qdrant는 Rust 기반 고성능 벡터 DB로 GPU 가속 지원 및 필터링 기능이 강점입니다. 성능 비교 실험 시 인덱싱 속도, 검색 응답 속도, 메모리 사용량, 스케일링 용이성 등을 비교합니다. 파이썬 코드 예제와 벤치마크 결과가 공개되어 있어, 개발자가 선택하기 용이합니다.

D2:['음식', '비빔밥', '역사']
비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기·계란 등)을 올리고 고추장이나 간장을 섞어 먹습니다. 전주 비빔밥은 고명

In [11]:
# 예시 3. 특정 저자가 작성한 문서만 검색
query = '벡터 데이터베이스 비교'

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        'author':{'$eq' : '김철수'}
    }
)

for doc in results:
    print(f"{doc.metadata['doc_id']}:{doc.metadata['author']}")
    print(doc.page_content)
    print()

D27:김철수
ChromaDB와 Qdrant는 벡터 검색 라이브러리로, ChromaDB는 오픈소스 벡터 DB로 간단한 파이썬 인터페이스를 제공하며, Qdrant는 Rust 기반 고성능 벡터 DB로 GPU 가속 지원 및 필터링 기능이 강점입니다. 성능 비교 실험 시 인덱싱 속도, 검색 응답 속도, 메모리 사용량, 스케일링 용이성 등을 비교합니다. 파이썬 코드 예제와 벤치마크 결과가 공개되어 있어, 개발자가 선택하기 용이합니다.

D29:김철수
Self-Query Retriever는 문서 메타데이터(제목·요약·키워드)를 분석해, 사용자가 실제로 검색할 만한 쿼리를 GPT-4o-mini 등 생성형 모델로 생성한 뒤, 생성된 가상 쿼리를 다시 검색에 활용하는 기법입니다. 이 과정을 통해 사용자가 입력한 실제 질의보다 검색 품질을 높이는 효과를 얻을 수 있으며, 생성된 쿼리는 ‘Self-Query’라고 불립니다.

D28:김철수
Contextual Compression은 긴 텍스트에서 핵심 정보만 추출해 압축(요약)한 뒤 검색 효율을 높이는 기법입니다. 예를 들어, 긴 문서를 PEGASUS 기반 한국어 요약 모델로 요약한 뒤, 압축된 요약문을 임베딩해 검색하면 문서 길이가 길어도 핵심 검색 품질을 유지할 수 있습니다. 압축 전후 검색 성능 차이를 평가할 때 Precision@k, Recall@k, nDCG@k 지표를 사용합니다.

D30:김철수
Multi-Hop Retrieval은 한 단계의 검색으로 해결되지 않는 복합 질문에 대응하기 위해, 여러 단계(홉)로 나눠서 검색을 수행하는 기법입니다. 예를 들어, “세종대왕이 훈민정음을 만든 이유를 바탕으로 AI 윤리 가이드라인 사례를 찾고 싶다”는 질문에서, 1단계로 ‘세종대왕 훈민정음 배경’(→D4 자료), 2단계로 ‘AI 윤리 가이드라인 사례’(→D7 또는 D25)로 연결해 답을 도출합니다. 단계별 결과를 종합해 최종 순위를 매깁니다.



In [12]:
# 예시 4. 특정 저자가 작성하지 않은 문서만 검색
query = '인공지능 윤리'

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        'author':{'$ne' : '김철수'}
    }
)

for doc in results:
    print(f"{doc.metadata['doc_id']}:{doc.metadata['author']}")
    print(doc.page_content)
    print()

D25:한지민
2025년 한국 정부는 “AI 국가전략 2.0”을 발표하며, 인공지능 기술 연구·개발(R&D) 예산을 3조 원으로 확대했습니다. 주요 정책으로는 AI 인재 양성, 규제 샌드박스 활성화, AI 윤리 가이드라인 강화, 공공데이터 개방 등이 포함됩니다. 특히 중소기업 대상 AI 솔루션 지원 사업이 증가하였고, 대학·연구기관 협력 프로젝트가 활발히 추진되고 있습니다.

D7:김도윤
최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발이 활발합니다. GPT 계열(예: GPT-4, GPT-4o-mini)은 자연어 생성·이해 능력이 뛰어나며, DALL·E, Stable Diffusion은 이미지 생성, CLIP 등은 이미지·텍스트 융합 모델로 주목받고 있습니다. 또한 AI 윤리 이슈로는 데이터 편향, 프라이버시 침해, 자율성 문제 등이 논의되고 있습니다.

D14:한지민
AI를 활용한 기후 예측 연구는 기계학습 모델을 통해 대규모 기상 데이터를 분석해 미래 기온·강수량을 예측합니다. 예를 들어, 한국 기상청과 KAIA가 공동으로 Deep Learning 기반 단기 기상 예보 모델을 개발했습니다. 강화학습을 적용해 극한 기상 상황 발생 확률을 시뮬레이션하는 연구도 진행 중이며, 기후 변화 대응 정책 수립에 활용되고 있습니다.

D11:이서연
건강을 위해서는 규칙적 식습관, 적절한 운동(주 3회 이상, 유산소+근력), 충분한 수면(하루 7~8시간), 스트레스 관리(명상·취미활동), 정기 건강검진이 필요합니다. 특히 비만 예방을 위해 저탄수화물·고단백 식단, 주간 10,000보 걷기를 권장하며, 음주·흡연은 최소화해야 합니다. 또한 정신 건강을 위해 긍정적 마인드, 사회적 지지 체계 구축, 전문 상담 서비스 이용도 도움이 됩니다.

D26:한지민
딥러닝 모델(예: LSTM, Transformer 기반 시계열 모델)을 활용한 기상 예측은 과거 기상 데이터로부터 패턴을 학습해 미래 기온·강수량을 예측합니다. 한국 기상청의 Deep Lea

In [13]:
# 예시 5. 여러 조건을 동시에 만족하는 문서 검색
query = '문법 검색 방법'

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        '$and':[
            {'author':{"$eq":"김철수"}},
            {'category':{"$in":["검색"]}},
        ]
    }
)

for doc in results:
    print(f"{doc.metadata['doc_id']}:{doc.metadata['author']}")
    print(doc.metadata['category'])
    print(doc.page_content)
    print()

D30:김철수
['검색', 'RAG', '멀티홉']
Multi-Hop Retrieval은 한 단계의 검색으로 해결되지 않는 복합 질문에 대응하기 위해, 여러 단계(홉)로 나눠서 검색을 수행하는 기법입니다. 예를 들어, “세종대왕이 훈민정음을 만든 이유를 바탕으로 AI 윤리 가이드라인 사례를 찾고 싶다”는 질문에서, 1단계로 ‘세종대왕 훈민정음 배경’(→D4 자료), 2단계로 ‘AI 윤리 가이드라인 사례’(→D7 또는 D25)로 연결해 답을 도출합니다. 단계별 결과를 종합해 최종 순위를 매깁니다.

D28:김철수
['검색', 'RAG', '압축']
Contextual Compression은 긴 텍스트에서 핵심 정보만 추출해 압축(요약)한 뒤 검색 효율을 높이는 기법입니다. 예를 들어, 긴 문서를 PEGASUS 기반 한국어 요약 모델로 요약한 뒤, 압축된 요약문을 임베딩해 검색하면 문서 길이가 길어도 핵심 검색 품질을 유지할 수 있습니다. 압축 전후 검색 성능 차이를 평가할 때 Precision@k, Recall@k, nDCG@k 지표를 사용합니다.

D29:김철수
['검색', 'RAG', '메타데이터']
Self-Query Retriever는 문서 메타데이터(제목·요약·키워드)를 분석해, 사용자가 실제로 검색할 만한 쿼리를 GPT-4o-mini 등 생성형 모델로 생성한 뒤, 생성된 가상 쿼리를 다시 검색에 활용하는 기법입니다. 이 과정을 통해 사용자가 입력한 실제 질의보다 검색 품질을 높이는 효과를 얻을 수 있으며, 생성된 쿼리는 ‘Self-Query’라고 불립니다.

D27:김철수
['검색', '벡터DB', '기술']
ChromaDB와 Qdrant는 벡터 검색 라이브러리로, ChromaDB는 오픈소스 벡터 DB로 간단한 파이썬 인터페이스를 제공하며, Qdrant는 Rust 기반 고성능 벡터 DB로 GPU 가속 지원 및 필터링 기능이 강점입니다. 성능 비교 실험 시 인덱싱 속도, 검색 응답 속도, 메모리 사용량, 스케일링 용이성 등을 비교합니다. 파이썬 코드 예

In [14]:
# 예시 6. 여러 조건 중 하나라도 만족하는 문서 검색
query = 'AI 검색 기술'

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        '$or':[
            {'author':{"$eq":"김철수"}},
            {'category':{"$in":["AI"]}},
        ]
    }
)

for doc in results:
    print(f"{doc.metadata['doc_id']}:{doc.metadata['author']}")
    print(doc.metadata['category'])
    print(doc.page_content)
    print()

D30:김철수
['검색', 'RAG', '멀티홉']
Multi-Hop Retrieval은 한 단계의 검색으로 해결되지 않는 복합 질문에 대응하기 위해, 여러 단계(홉)로 나눠서 검색을 수행하는 기법입니다. 예를 들어, “세종대왕이 훈민정음을 만든 이유를 바탕으로 AI 윤리 가이드라인 사례를 찾고 싶다”는 질문에서, 1단계로 ‘세종대왕 훈민정음 배경’(→D4 자료), 2단계로 ‘AI 윤리 가이드라인 사례’(→D7 또는 D25)로 연결해 답을 도출합니다. 단계별 결과를 종합해 최종 순위를 매깁니다.

D29:김철수
['검색', 'RAG', '메타데이터']
Self-Query Retriever는 문서 메타데이터(제목·요약·키워드)를 분석해, 사용자가 실제로 검색할 만한 쿼리를 GPT-4o-mini 등 생성형 모델로 생성한 뒤, 생성된 가상 쿼리를 다시 검색에 활용하는 기법입니다. 이 과정을 통해 사용자가 입력한 실제 질의보다 검색 품질을 높이는 효과를 얻을 수 있으며, 생성된 쿼리는 ‘Self-Query’라고 불립니다.

D14:한지민
['AI', '기후', '연구']
AI를 활용한 기후 예측 연구는 기계학습 모델을 통해 대규모 기상 데이터를 분석해 미래 기온·강수량을 예측합니다. 예를 들어, 한국 기상청과 KAIA가 공동으로 Deep Learning 기반 단기 기상 예보 모델을 개발했습니다. 강화학습을 적용해 극한 기상 상황 발생 확률을 시뮬레이션하는 연구도 진행 중이며, 기후 변화 대응 정책 수립에 활용되고 있습니다.

D28:김철수
['검색', 'RAG', '압축']
Contextual Compression은 긴 텍스트에서 핵심 정보만 추출해 압축(요약)한 뒤 검색 효율을 높이는 기법입니다. 예를 들어, 긴 문서를 PEGASUS 기반 한국어 요약 모델로 요약한 뒤, 압축된 요약문을 임베딩해 검색하면 문서 길이가 길어도 핵심 검색 품질을 유지할 수 있습니다. 압축 전후 검색 성능 차이를 평가할 때 Precision@k, Recall@k, nDCG@k 지표를 사용